# 2c - Minimum risk, with AI

**Time:** 45 minutes build + 20 minutes check  |  **AI: writes code from YOUR spec** - you explain every line

**What you will do**
- Write a spec (a short written brief of what you want): Goal, Inputs, Units and conventions, Output, Check.
- Ask Gemini to write the code from your spec. Paste it here.
- Explain every line in your own words.
- Then check the answer, with plain loops and with your own eyes.

The code is written by AI. The decision is yours: what you ask for decides what you get.

## The problem

Same client funds A, B and C, same vols and correlations as in 2a and 2b. Run the code cell below first: it loads the data and your function.

**Find the portfolio of A, B and C with the lowest risk.**

First, a quick reminder of how a good spec looks. This is the spec for `portfolio_vol`, the function you built in 2b:

| | The example from 2b |
|---|---|
| **Goal** | Volatility of a portfolio of funds |
| **Inputs** | weights, vols, correlation matrix (arrays) |
| **Units and conventions** | decimals (0.15 = 15%), annual, weights sum to 1 |
| **Output** | one number, the annual volatility |
| **Check** | all correlations = 1 gives the weighted average of the vols |

In [ ]:
import numpy as np

# Given: the same data as in 2b
vol = np.array([0.15, 0.20, 0.25])    # volatilities of A, B, C
corr = np.array([[1.0, 0.1, 0.4],
                 [0.1, 1.0, 0.7],
                 [0.4, 0.7, 1.0]])


# Given: your function from 2b (you will need it in the check)
def portfolio_vol(weights, vol, corr):
    if round(np.sum(weights), 6) != 1:
        raise ValueError("Weights must sum to 1.")
    cov = np.diag(vol) @ corr @ np.diag(vol)
    return np.sqrt(weights @ cov @ weights)

## Step 1 - Write your spec

Double-click this cell and replace each `...` with your text, for the minimum-risk problem above. Press Shift+Enter to close it. Be precise: AI fills every gap with its own guess.

| | Your spec |
|---|---|
| **Goal** | ... |
| **Inputs** | ... |
| **Units and conventions** | ... |
| **Output** | ... |
| **Check** | ... |

## Step 2 - Ask Gemini, paste the code

1. Open Gemini (gemini.google.com, or the Gemini button in Colab).
2. Paste your spec, then ask: *"Write Python code with NumPy for this spec."*
3. Paste the code in the cell below.
4. Make the code leave the weights in a variable called `ai_weights` (an array of 3 numbers), and print the weights and the volatility. If Gemini gave other names, rename them.

**Stuck?** If the code fails, or there is no `ai_weights` after you run it, do not skip the checks. Ask your neighbour, or run the backup cell below.

Run the cell.

In [ ]:
# PASTE the code Gemini wrote here.
# The inputs vol and corr are defined above. Keep the variable name ai_weights.

### Backup: a classmate's AI answer

Run the cell below **only if** your own cell above did not work, or did not create `ai_weights`. It is the code that most AI tools write for this problem. You still explain every line in step 3.

If your own code works, skip this cell for now. You come back to it at the end.

In [ ]:
# BACKUP ONLY: a classmate's AI answer. Run it only if your own code did not work.
cov = np.diag(vol) @ corr @ np.diag(vol)      # covariance matrix, as in 2a
inv_cov = np.linalg.inv(cov)                  # the inverse of the covariance matrix (new, not part of the course)
ones = np.ones(3)                             # one 1 per fund
ai_weights = inv_cov @ ones / (ones @ inv_cov @ ones)   # the textbook minimum-variance formula
ai_vol = np.sqrt(ai_weights @ cov @ ai_weights)

print("Weights:", ai_weights)
print(f"Volatility: {ai_vol:.2%}")

## Step 3 - Explain every line

Rule of the course: you must be able to explain every line you submit. For each line of the code above, write what it does, in your own words and in finance words.

Double-click here and write: ...

## Check - Is the answer right?

AI wrote code, and it ran. That is not a check. Three questions, then a brute force.

(Checks card, entries 7 and 8: see the link at the bottom of this notebook.)

### Question 1 - Do the weights sum to 1?

Add up the weights in `ai_weights` with `np.sum`.

In [ ]:
total = ...   # YOUR CODE
print(f"Sum of the weights: {total:.4f}")

### Question 2 - Is the volatility below the lowest single fund?

The lowest single fund is A, at 15%. A portfolio of the three funds should do at least as well.

Call your function `portfolio_vol` from 2b on `ai_weights`, with `vol` and `corr`.

In [ ]:
check_vol = ...   # YOUR CODE
print(f"Volatility: {check_vol:.2%}")
print("Below 15%?", check_vol < 0.15)

### Question 3 - Look at each weight

Do not only check the total. Look at every weight, one by one. A weight is a position in a fund: a positive weight is a purchase, a negative weight is a short position (a bet that the fund will fall).

In [ ]:
print(f"Fund A: {ai_weights[0]:.1%}")
print(f"Fund B: {ai_weights[1]:.1%}")
print(f"Fund C: {ai_weights[2]:.1%}")

**Your answer:** is any weight negative? If yes, what does a negative weight mean in real life? Can this client do that? Did your spec say so? Write 3 to 4 lines here: ...

### Brute force: try every long-only portfolio

A second method, **brute force**: try every possibility and keep the best. No clever formula, only plain loops from Session 1. Try every portfolio in steps of 1% with no negative weights (this is called **long-only**). `a` is the percent in A (0 to 100), `b` the percent in B (0 to `100 - a`), and the rest goes in C. That is 5,151 portfolios. Keep the one with the lowest volatility, using your `portfolio_vol`.

For each pair `a`, `b`: the percent in C is what is left of 100 after A and B. Get the volatility of the trial weights with `portfolio_vol` (with `vol` and `corr`). If it beats the best volatility so far, keep both that volatility and the trial weights that gave it.

You should get **65 / 35 / 0**, volatility **12.56%**, 5,151 portfolios.

In [ ]:
best_vol = 1       # start higher than any real vol (100%)
best_w = None      # None means: nothing yet
count = 0

for a in range(0, 101):
    for b in range(0, 101 - a):
        c_pct = ...   # YOUR CODE
        trial = np.array([a, b, c_pct]) / 100    # percent -> decimals
        v = ...       # YOUR CODE
        count = count + 1
        if v < best_vol:
            best_vol = ...   # YOUR CODE
            best_w = ...     # YOUR CODE

print(f"Portfolios tried: {count:,}")
print(f"Best weights: A {best_w[0]:.0%}, B {best_w[1]:.0%}, C {best_w[2]:.0%}")
print(f"Best volatility: {best_vol:.2%}")

### Your conclusion

The AI may say one thing, the brute force another. **Why do the two answers differ?** What should your spec have said? Write 3 to 4 lines here: ...

Note: if your AI's answer has no negative weight, do not stop here. Compare it with the brute force, and check what method and what constraint it used. Then run the backup cell (a classmate's answer) and run Question 3 again: same problem, a different spec, a different answer. Which one would you send to the MD?

## What you learned

**Python**
- **Function**: `portfolio_vol(weights, vol, corr)` reused inside a loop, with its **Guard** (`raise ValueError`) still checking every portfolio.
- **Array** and **Matrix product** in AI code: `inv_cov @ ones`, `ones @ inv_cov @ ones`. **Diagonal matrix** (`np.diag`), **Square root** (`np.sqrt`), **2D array + index** and **Shape** all appear in the code you were asked to read.
- **For loop** inside a for loop, with `range` and `if v < best_vol`, to try 5,151 portfolios in plain Python.
- **Import**: `import numpy as np`. Indexing `ai_weights[2]` reads one weight.

**Finance**
- A spec is five lines: Goal, Inputs, Units and conventions, Output, Check. AI fills every gap with its own guess.
- Weights that sum to 1 and a low volatility are not enough. A negative weight is a short position: check the constraints, not only the number.
- The clever method (72 / 50 / -22, vol 12.0%) and the brute force (65 / 35 / 0, vol 12.56%) disagree. The spec decides the answer.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right